# Pose-based Saudi Sign Language Recognition

Translating continuous Saudi Sign Language into Arabic gloss sequences from skeleton keypoints. This notebook carries the shared data pipeline and two of the four architectures compared in the project: the recurrent baseline and the model that produced the final result.

## What is in here

**Data and preprocessing.** Loading the Isharah 1000 signer-independent split, reconciling the gloss CSVs against the pose records, turning each frame of keypoints into a feature vector, and building the gloss vocabulary. Everything downstream depends on this section.

**BiLSTM with CTC, the baseline.** A bidirectional recurrent model over flattened per-frame features. Best dev WER **66.62%**.

**Conformer with CTC.** Self-attention for long-range structure paired with convolution for local motion. Best dev WER **13.04%**, an 80% relative reduction on identical inputs, and the only model taken through to the 3,800-sentence test set.

The other two architectures live in their own notebooks: `02-transformer-ctc.ipynb` and `03-conformer-seq2seq.ipynb`.

## The data

**Input.** 86 keypoints per frame in two dimensions, covering the upper body skeleton, both hands, the face and the lip contour. There is no depth channel. Per-frame velocities are appended, giving 344 features per frame.

**Target.** A sequence of Arabic gloss tokens, 675 distinct words in the corpus.

**Task.** Continuous sign language recognition, trained with CTC loss, which assumes the output is monotonically aligned with the input. That assumption holds here: signs are produced in order.

Authors: Mohammed Al Sheqaih, Abdulrahman Ammar, Naif Alenazi
Supervisor: Dr. Hamzah Luqman

In [1]:
# Core libraries
import os
import random
import numpy as np
import pandas as pd
import pickle
import torch

### 1. Data Loading and Inspection

In this first part, the goal is simply to open the CSV files and the pose (.pkl file), making sure that all IDs line up, and get a concrete feeling for shapes and basic statistics (number of clips, average frames, gloss lengths, etc.). This will guide every design choice later (batching, model depth, etc.) and keeps us from guessing about the data layout. The Isharah track uses pose data with 86 keypoints per frame for ~14k videos and 1k sentences, so confirming that this is what we actually see on disk is important.

#### 1.1 Mounting the drive and checking current directory

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("Current working directory:", os.getcwd())

Mounted at /content/drive
Current working directory: /content


#### 1.2. Load train/dev CSV files

In [3]:
import os
import pandas as pd

# Folder you created in Google Drive
data_dir = "/content/drive/MyDrive/IsharahData"

train_csv_path = os.path.join(data_dir, "train.csv")
dev_csv_path   = os.path.join(data_dir, "dev.csv")
pose_path      = os.path.join(data_dir, "pose_data_isharah1000_hands_lips_body_May12.pkl")


print("Train exists:", os.path.exists(train_csv_path))
print("Dev exists:  ", os.path.exists(dev_csv_path))
print("Pose exists: ", os.path.exists(pose_path))


# Read the CSVs; encoding='utf-8' is usually fine for Arabic glosses
train_df = pd.read_csv(train_csv_path)
dev_df   = pd.read_csv(dev_csv_path)

print("Train shape:", train_df.shape)
print("Dev shape  :", dev_df.shape)

# Quick peek at a few rows to see how the glosses look
train_df.head()

# Sanity check: the expected columns should be exactly ['id', 'gloss']
print("Train columns:", list(train_df.columns))
print("Dev columns  :", list(dev_df.columns))

# Print a couple of random examples just to get a feeling for the data
train_df.sample(3, random_state=42)

Train exists: True
Dev exists:   True
Pose exists:  True
Train shape: (10000, 2)
Dev shape  : (949, 2)
Train columns: ['id', 'gloss']
Dev columns  : ['id', 'gloss']


,id,gloss
6252,03_0753,طفل طفل بكاء سبب جوع
4684,14_0185,رغبه صوره قط صغير
1731,04_0232,هو دفع مال سوال


In [4]:
# Load test pose .pkl
BASE_PATH = "/content/drive/MyDrive/IsharahData"

import pickle, pandas as pd

test_pose_pkl_path = f"{BASE_PATH}/pose_data_isharah1000_SI_test.pkl"
with open(test_pose_pkl_path, "rb") as f:
    test_pose_data = pickle.load(f)

# Create test.csv from the .pkl keys
test_ids = sorted(list(test_pose_data.keys()))
test_df = pd.DataFrame({"id": test_ids, "gloss": [""] * len(test_ids)})
test_csv_path = f"{BASE_PATH}/test.csv"
test_df.to_csv(test_csv_path, index=False, encoding="utf-8")



/tmp/ipython-input-1275223075.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_pose_data = pickle.load(f)


#### 1.3. Load the pose pickle and inspect its structure

In [5]:
import pickle

pose_filename = "pose_data_isharah1000_hands_lips_body_May12.pkl"
pose_path = os.path.join(data_dir, pose_filename)

with open(pose_path, "rb") as f:
    pose_data = pickle.load(f)

print("Type of pose_data object:", type(pose_data))
print("Number of sequence entries in pose_data:", len(pose_data))

/tmp/ipython-input-706123235.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  pose_data = pickle.load(f)


Type of pose_data object: <class 'dict'>
Number of sequence entries in pose_data: 10450


In [6]:
# We Look inside a single entry to understand what is stored there
example_key = next(iter(pose_data.keys()))
example_value = pose_data[example_key]

print("Example ID from pose_data:", example_key)
print("Type of pose_data[ID]:", type(example_value))

if isinstance(example_value, dict):
    print("\nInner keys for this ID and their shapes:")
    for k, v in example_value.items():
        shape = getattr(v, "shape", None)
        print(f"  - {k!r}: type={type(v)}, shape={shape}")
else:
    print("This entry is not a dict, which is unexpected for this dataset.")

Example ID from pose_data: 00_0001
Type of pose_data[ID]: <class 'dict'>

Inner keys for this ID and their shapes:
  - 'keypoints': type=<class 'numpy.ndarray'>, shape=(55, 86, 2)


#### 1.4. Choose the pose array key and define helpers

In [7]:
import numpy as np

# From the inspection above, every entry looks like: {'keypoints': np.ndarray[T, 86, 2]}
POSE_ARRAY_KEY = "keypoints"

def get_pose_array(seq_id):
    """
    Fetch the raw pose array for a given sequence ID as a NumPy array.
    For this dataset, the array should have shape [T, 86, 2].
    """
    item = pose_data[str(seq_id)]
    raw = item[POSE_ARRAY_KEY]
    arr = np.asarray(raw, dtype=np.float32)
    return arr

# Quick check on a few training IDs just to be sure everything is consistent
for sid in train_df["id"].head(3):
    arr = get_pose_array(sid)
    print(f"ID={sid}, pose shape={arr.shape}")

ID=00_0001, pose shape=(55, 86, 2)
ID=00_0002, pose shape=(143, 86, 2)
ID=00_0003, pose shape=(98, 86, 2)


#### 1.5. Consistency check, filter invalid train IDs, then stats


In [8]:
# Make sure every train/dev ID has a corresponding pose entry
train_ids = set(train_df["id"].astype(str))
dev_ids   = set(dev_df["id"].astype(str))
pose_ids  = set(map(str, pose_data.keys()))

missing_in_pose_train = train_ids - pose_ids
missing_in_pose_dev   = dev_ids - pose_ids

print("Train IDs missing in pose_data:", len(missing_in_pose_train))
print("Dev IDs missing in pose_data  :", len(missing_in_pose_dev))

if missing_in_pose_train:
    print("Example missing train ID:", next(iter(missing_in_pose_train)))
if missing_in_pose_dev:
    print("Example missing dev ID  :", next(iter(missing_in_pose_dev)))

Train IDs missing in pose_data: 500
Dev IDs missing in pose_data  : 0
Example missing train ID: 13_0465


In [9]:
# Since some training IDs have no pose data, I will drop those rows from train_df.
# There is no point in keeping examples that I cannot feed to the model.
valid_train_mask = train_df["id"].astype(str).isin(pose_ids)
filtered_train_df = train_df[valid_train_mask].reset_index(drop=True)

num_dropped = len(train_df) - len(filtered_train_df)
print(f"Original train size: {len(train_df)}")
print(f"Filtered  train size: {len(filtered_train_df)}")
print(f"Dropped rows without pose data: {num_dropped}")

# From now on, I will overwrite train_df with the filtered version
train_df = filtered_train_df

Original train size: 10000
Filtered  train size: 9500
Dropped rows without pose data: 500


Now train_df only contains IDs that exist in pose_data, so get_pose_array will no longer raise KeyError.

In [10]:
# Helper: how many frames does this sequence have?
def get_seq_len(seq_id):
    return get_pose_array(seq_id).shape[0]

train_frame_lengths = [get_seq_len(sid) for sid in train_df["id"]]
dev_frame_lengths   = [get_seq_len(sid) for sid in dev_df["id"]]

print("Frames per clip (train): min =", min(train_frame_lengths),
      ", max =", max(train_frame_lengths),
      ", mean =", float(np.mean(train_frame_lengths)))

print("Frames per clip (dev):   min =", min(dev_frame_lengths),
      ", max =", max(dev_frame_lengths),
      ", mean =", float(np.mean(dev_frame_lengths)))

Frames per clip (train): min = 33 , max = 788 , mean = 213.96694736842105
Frames per clip (dev):   min = 61 , max = 767 , mean = 326.95468914646995


In [11]:
# Very simple sentence-length stats (number of tokens per gloss)
def gloss_len(text):
    return len(str(text).split())

train_gloss_lens = [gloss_len(g) for g in train_df["gloss"]]
dev_gloss_lens   = [gloss_len(g) for g in dev_df["gloss"]]

print("Gloss length (train): min =", min(train_gloss_lens),
      ", max =", max(train_gloss_lens),
      ", mean =", float(np.mean(train_gloss_lens)))

print("Gloss length (dev):   min =", min(dev_gloss_lens),
      ", max =", max(dev_gloss_lens),
      ", mean =", float(np.mean(dev_gloss_lens)))

Gloss length (train): min = 1 , max = 12 , mean = 4.798315789473684
Gloss length (dev):   min = 1 , max = 12 , mean = 4.798735511064278


### 2. Pose preprocessing and gloss vocabulary

In this part, we define a simple yet consistent way to convert the raw pose arrays into feature sequences and build a vocabulary over gloss tokens. The pose features will be used as input to both models, and the gloss vocabulary will be shared by the CTC loss and the decoding step.

#### 2.1. Pose to a feature sequence

In [12]:
import numpy as np

# Define keypoint groups based on Isharah's 86-keypoint layout
# Typically: body (11), left hand (21), right hand (21), face/lips (33)
BODY_INDICES = list(range(0, 11))
LEFT_HAND_INDICES = list(range(11, 32))
RIGHT_HAND_INDICES = list(range(32, 53))
FACE_INDICES = list(range(53, 86))

# Most important keypoints for sign language (hands dominate)
PRIORITY_INDICES = LEFT_HAND_INDICES + RIGHT_HAND_INDICES + BODY_INDICES[:5]

def pose_to_feature_sequence(arr: np.ndarray,
                              use_velocity: bool = True,
                              use_acceleration: bool = False,
                              filter_keypoints: bool = False) -> np.ndarray:
    """
    Enhanced pose preprocessing for skeleton-based CSLR.

    Features:
    - Body-centric normalization (translation invariant)
    - Scale normalization (signer size invariant)
    - Velocity features (motion dynamics)
    - Optional acceleration features
    - Optional keypoint filtering (hands + upper body only)
    """
    arr = np.asarray(arr, dtype=np.float32)

    if arr.ndim == 2:
        # Already flattened, reshape assuming 86 joints × 2 coords
        T, D = arr.shape
        arr = arr.reshape(T, D // 2, 2)

    T, J, C = arr.shape  # [T, 86, 2]

    # 1. Optional: Filter to priority keypoints (reduces noise from face micro-movements)
    if filter_keypoints and J == 86:
        arr = arr[:, PRIORITY_INDICES, :]  # [T, ~47, 2]
        J = len(PRIORITY_INDICES)

    # 2. Body-centric centering (use body center or mean of shoulders)
    # This makes the representation translation-invariant
    if J == 86 or not filter_keypoints:
        center = arr[:, BODY_INDICES, :].mean(axis=1, keepdims=True)  # body center
    else:
        center = arr.mean(axis=1, keepdims=True)
    arr_centered = arr - center  # [T, J, 2]

    # 3. Scale normalization (makes representation size-invariant)
    # Use the standard deviation of all coordinates as scale factor
    scale = arr_centered.std() + 1e-6
    arr_norm = arr_centered / scale  # [T, J, 2]

    # 4. Flatten to [T, J*C]
    feat = arr_norm.reshape(T, J * C)

    features = [feat]

    # 5. Velocity features (first derivative) - CRITICAL for motion
    if use_velocity:
        velocity = np.zeros_like(feat)
        velocity[1:] = feat[1:] - feat[:-1]
        features.append(velocity)

    # 6. Acceleration features (second derivative) - optional, adds more dynamics
    if use_acceleration:
        acceleration = np.zeros_like(feat)
        acceleration[2:] = velocity[2:] - velocity[1:-1]
        features.append(acceleration)

    # Concatenate all features along feature dimension
    final_feat = np.concatenate(features, axis=-1)  # [T, D'] where D' = J*C * num_features

    return final_feat.astype(np.float32)


In [13]:
def augment_pose_sequence(arr: np.ndarray, training: bool = True) -> np.ndarray:
    """
    Comprehensive augmentation for pose sequences.
    Applied BEFORE pose_to_feature_sequence.
    """
    if not training:
        return arr

    arr = arr.copy()
    T, J, C = arr.shape

    # 1. Temporal augmentation: random speed variation (very effective)
    if np.random.rand() < 0.5:
        speed_factor = np.random.uniform(0.8, 1.2)
        new_T = max(int(T * speed_factor), 10)  # at least 10 frames
        old_indices = np.arange(T)
        new_indices = np.linspace(0, T - 1, new_T)
        # Linear interpolation for smooth resampling
        arr_resampled = np.zeros((new_T, J, C), dtype=np.float32)
        for j in range(J):
            for c in range(C):
                arr_resampled[:, j, c] = np.interp(new_indices, old_indices, arr[:, j, c])
        arr = arr_resampled
        T = new_T

    # 2. Spatial scaling (body size variation)
    if np.random.rand() < 0.5:
        scale = np.random.uniform(0.85, 1.15)
        arr = arr * scale

    # 3. Random rotation (simulates slight camera angle changes)
    if np.random.rand() < 0.5: # NEW: Increased from 0.4
        angle = np.random.uniform(-12, 12) * np.pi / 180  # ±12 degrees #: Reduced from 20 to 12
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        rotation_matrix = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float32)
        # Apply rotation: [T, J, 2] @ [2, 2] -> [T, J, 2]
        arr = np.einsum('tjc,cd->tjd', arr, rotation_matrix)

    # 4. Gaussian noise (simulates pose estimation noise)
    if np.random.rand() < 0.5:
        noise_std = np.random.uniform(0.005, 0.02)
        noise = np.random.normal(0, noise_std, arr.shape).astype(np.float32)
        arr = arr + noise

    # 5. Random temporal crop (use 80-100% of the sequence)
    if np.random.rand() < 0.3 and T > 30:
        crop_ratio = np.random.uniform(0.8, 1.0)
        crop_len = int(T * crop_ratio)
        start_idx = np.random.randint(0, T - crop_len + 1)
        arr = arr[start_idx:start_idx + crop_len]

    # 6. Joint dropout (randomly zero out some joints - regularization)
    if np.random.rand() < 0.3: # better than 0.2
        num_drop = np.random.randint(1, min(5, J))
        drop_indices = np.random.choice(J, num_drop, replace=False)
        arr[:, drop_indices, :] = 0.0

    # 7. Frame Masking (forces context learning) #### NEW: ADDED FOR TRIAL AFTER THE FIRST TRAINING
    if np.random.rand() < 0.5:
        mask_len = np.random.randint(1, 10)  # Mask 1-10 consecutive frames
        num_masks = int(T * 0.1) // mask_len # Mask roughly 10% of time
        for _ in range(num_masks):
            t_start = np.random.randint(0, T - mask_len)
            arr[t_start:t_start+mask_len, :, :] = 0.0

    return arr.astype(np.float32)

In [14]:
example_arr = get_pose_array(train_df["id"].iloc[0])
example_feat = pose_to_feature_sequence(example_arr)
print("Raw pose shape:", example_arr.shape)
print("Feature sequence shape:", example_feat.shape)

Raw pose shape: (55, 86, 2)
Feature sequence shape: (55, 344)


#### 2.2. Build gloss vocabulary
We build a simple word‑level vocabulary from train + dev glosses. CTC will handle blanks internally; we still need < pad > and < unk > for batching and robustness.


In [15]:
from collections import Counter

# Collect all gloss tokens from train and dev
all_glosses = list(train_df["gloss"]) + list(dev_df["gloss"])

token_counter = Counter()
for g in all_glosses:
    tokens = str(g).split()
    token_counter.update(tokens)

print("Total unique gloss tokens:", len(token_counter))

Total unique gloss tokens: 675


In [16]:
# Reserve a few special tokens; blank for CTC is handled separately by PyTorch
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

# Start the vocab with special tokens
id2token = [PAD_TOKEN, UNK_TOKEN]
token2id = {PAD_TOKEN: 0, UNK_TOKEN: 1}

for tok, _ in token_counter.most_common():
    if tok not in token2id:
        token2id[tok] = len(id2token)
        id2token.append(tok)

vocab_size = len(id2token)
print("Final vocab size (including specials):", vocab_size)

Final vocab size (including specials): 677


In [17]:
def encode_gloss(text: str):
    """
    Convert a gloss sentence into a list of integer token IDs.
    Unknown tokens fall back to <unk>.
    """
    tokens = str(text).split()
    ids = [token2id.get(t, token2id[UNK_TOKEN]) for t in tokens]
    return ids

# Quick sanity check on a random training example
sample_row = train_df.sample(1, random_state=7).iloc[0]
print("Sample gloss:", sample_row["gloss"])
print("Encoded as IDs:", encode_gloss(sample_row["gloss"]))
print("Decoded back:", " ".join(id2token[i] for i in encode_gloss(sample_row["gloss"])))

Sample gloss: تقديم حديقه فلاح سله جزر انواع
Encoded as IDs: [142, 140, 485, 667, 668, 86]
Decoded back: تقديم حديقه فلاح سله جزر انواع


### 3. PyTorch Dataset and collate function
Here we wrap everything into a PyTorch Dataset so that we can easily batch and shuffle examples during training. The dataset returns a normalized pose feature sequence along with its length and the encoded gloss token sequence. The custom collate_fn pads sequences in a batch to the same length, which keeps the rest of the training code cleaner.

#### 3.1. Dataset class

In [18]:
from torch.utils.data import Dataset, DataLoader

class IsharahPoseDataset(Dataset):
    def __init__(self, df, pose_data, pose_key, token2id, training=False):
        """
        Args:
            df: DataFrame with 'id' and 'gloss' columns
            pose_data: dict mapping id -> {pose_key: np.ndarray}
            pose_key: key to access pose array (e.g., 'keypoints')
            token2id: vocabulary mapping
            training: if True, apply augmentation
        """
        self.df = df.reset_index(drop=True)
        self.pose_data = pose_data
        self.pose_key = pose_key
        self.token2id = token2id
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq_id = str(row["id"])
        gloss = row["gloss"]

        # Get raw pose [T, 86, 2]
        raw_pose = self.pose_data[seq_id][self.pose_key]
        pose_arr = np.asarray(raw_pose, dtype=np.float32)

        # Apply augmentation if training
        if self.training:
            pose_arr = augment_pose_sequence(pose_arr, training=True)

        # Convert to feature sequence [T, D]
        feat_seq = pose_to_feature_sequence(pose_arr, use_velocity=True)

        # Encode gloss
        target_ids = encode_gloss(gloss)

        # Convert to tensors
        feat_tensor = torch.from_numpy(feat_seq)  # [T, D]
        target_tensor = torch.tensor(target_ids, dtype=torch.long)

        sample = {
                "id": seq_id,
                "pose": feat_tensor,
                "pose_lens": torch.tensor(feat_tensor.shape[0], dtype=torch.long),
                "targets": target_tensor,
                "target_lens": torch.tensor(len(target_ids), dtype=torch.long),
                }

        return sample

#### 3.2. Collate function for batching

In [19]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    # IDs
    ids = [item["id"] for item in batch]

    # Pose sequences
    pose_seqs = [item["pose"] for item in batch]                  # list of [T_i, D]
    pose_lens = torch.stack([item["pose_lens"] for item in batch])  # [B]

    # Targets
    target_seqs = [item["targets"] for item in batch]             # list of [L_i]
    target_lens = torch.stack([item["target_lens"] for item in batch])  # [B]

    # Pad poses to [B, T_max, D]
    pose_padded = pad_sequence(pose_seqs, batch_first=True, padding_value=0.0)

    # Pad targets to [B, L_max] with PAD token
    pad_id = token2id[PAD_TOKEN]
    targets_padded = pad_sequence(
        target_seqs, batch_first=True, padding_value=pad_id
    )

    batch_out = {
        "id": ids,
        "pose": pose_padded,
        "pose_lens": pose_lens,
        "targets": targets_padded,
        "target_lens": target_lens,
    }
    return batch_out

#### 3.3. DataLoaders (Train, Dev)

In [49]:
# Dynamically compute input dimension after preprocessing changes
sample_arr = get_pose_array(train_df["id"].iloc[0])
sample_feat = pose_to_feature_sequence(sample_arr, use_velocity=True)
INPUT_DIM = sample_feat.shape[-1]
print(f"Input feature dimension: {INPUT_DIM}")  # Should be 344 with velocity

# Create datasets with training flag
train_dataset = IsharahPoseDataset(
    train_df, pose_data, POSE_ARRAY_KEY, token2id, training=True
)
dev_dataset = IsharahPoseDataset(
    dev_df, pose_data, POSE_ARRAY_KEY, token2id, training=False
)

# Create loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_batch,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_batch,
    pin_memory=True,
)

# Verify shapes
batch = next(iter(train_loader))
print(f"Batch pose shape: {batch['pose'].shape}")  # [B, Tmax, D]
print(f"Batch pose_lens: {batch['pose_lens']}")

Input feature dimension: 344
Batch pose shape: torch.Size([16, 402, 344])
Batch pose_lens: tensor([240, 123, 207, 177, 286, 127, 310, 219, 199, 307, 205, 164, 319, 402,
         83, 191])


#### 3.4 Dataset, Collate, and Dataloader (Test)

In [53]:
import pickle
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

# 1. Load Test Data
BASE_PATH = "/content/drive/MyDrive/IsharahData"

# Load test pose dictionary
test_pose_pkl_path = f"{BASE_PATH}/pose_data_isharah1000_SI_test.pkl"
with open(test_pose_pkl_path, "rb") as f:
    test_pose_data = pickle.load(f)
print(f"Loaded test pose data with {len(test_pose_data)} samples")

# 2. Create a test dataframe with empty glosses
test_ids = sorted(list(test_pose_data.keys()))
test_df = pd.DataFrame({
    "id": test_ids,
    "gloss": [""] * len(test_ids) # Empty gloss for test set
})
print("Test DataFrame created:", test_df.shape)

class IsharahTestDataset(Dataset):
    def __init__(self, df, pose_dict):
        self.df = df.reset_index(drop=True)
        self.pose_dict = pose_dict

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sid = row["id"]

        # Get the raw object from the pickle
        raw_pose = self.pose_dict[sid]

        # Handle if it's a dictionary (common in test sets)
        if isinstance(raw_pose, dict):
            # Try common keys used in Isharah datasets
            if "front" in raw_pose:
                pose_arr = raw_pose["front"]
            elif "pose" in raw_pose:
                pose_arr = raw_pose["pose"]
            else:
                # Fallback: take the first value if keys are unknown
                pose_arr = next(iter(raw_pose.values()))
        else:
            # It's already an array (like training data)
            pose_arr = raw_pose

        # Now convert to features
        feat_seq = pose_to_feature_sequence(pose_arr)
        feat_seq = feat_seq.astype(np.float32)

        return {
            "id": sid,
            "pose": torch.from_numpy(feat_seq),
            "pose_lens": torch.tensor(feat_seq.shape[0], dtype=torch.long),
        }

# 3. Define Test Collate Function
def collate_test_batch(batch):
    # Sort by length for efficient processing (optional but good practice)
    batch = sorted(batch, key=lambda x: x["pose_lens"], reverse=True)

    ids = [b["id"] for b in batch]
    lens = torch.stack([b["pose_lens"] for b in batch])

    B = len(batch)
    T_max = lens.max().item()
    F = batch[0]["pose"].size(-1)

    pose_padded = torch.zeros(B, T_max, F, dtype=torch.float32)
    for i, b in enumerate(batch):
        T = b["pose"].size(0)
        pose_padded[i, :T] = b["pose"]

    return {
        "id": ids,
        "pose": pose_padded,
        "pose_lens": lens
    }

# 4. Create the DataLoader
BATCH_SIZE_TEST = 16

test_dataset = IsharahTestDataset(test_df, test_pose_data)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE_TEST,
    shuffle=False,  # Important: keep order for submission
    num_workers=2,
    collate_fn=collate_test_batch
)

print("Test Loader created successfully.")


/tmp/ipython-input-148165588.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_pose_data = pickle.load(f)


Loaded test pose data with 3800 samples
Test DataFrame created: (3800, 2)
Test Loader created successfully.


### A shared config cell (FOR both models)

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Device setup (shared by both models)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Dynamically compute input dimension from a sample (shared)
sample_feat = pose_to_feature_sequence(get_pose_array(train_df["id"].iloc[0]))
INPUT_DIM = sample_feat.shape[-1]
print(f"Input dimension: {INPUT_DIM}")

# Vocabulary setup (shared)
# id2token should already be built earlier from your gloss vocabulary
VOCAB_SIZE = len(id2token)              # number of real gloss tokens
CTC_BLANK_INDEX = VOCAB_SIZE           # blank index is last
OUTPUT_DIM = VOCAB_SIZE + 1            # real tokens + blank
print(f"Vocab size: {VOCAB_SIZE}, CTC blank index: {CTC_BLANK_INDEX}")

# CTC loss (shared)
ctc_loss_fn = nn.CTCLoss(
    blank=CTC_BLANK_INDEX,
    zero_infinity=True
)

Using device: cuda
Input dimension: 344
Vocab size: 677, CTC blank index: 677


### 4. Model 1: BiLSTM + CTC

Our baseline architecture consists of a simple BiLSTM network. It processes the flattened pose vectors at each time step to capture sequential context.
- **Input:** Flattened pose vectors `(Batch, Time, Features)`
- **Core:** 3-layer BiLSTM
- **Output:** Linear projection to vocabulary size (CTC logits)


#### 4.1. Model configuration

In [23]:
# 4.1 Model configuration for BiLSTM + CTC (using shared config)

bilstm_config = {
    "input_dim": INPUT_DIM,     # from shared config
    "hidden_dim": 256,
    "num_layers": 3,
    "dropout": 0.3,
    "bidirectional": True,
    "vocab_size": OUTPUT_DIM    # same output dim as Conformer (tokens + blank)
}

bilstm_config

{'input_dim': 344,
 'hidden_dim': 256,
 'num_layers': 3,
 'dropout': 0.3,
 'bidirectional': True,
 'vocab_size': 678}

#### 4.2. BiLSTM model implementation

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BiLSTM_CTC(nn.Module):
    """
    Simple baseline: stacked BiLSTM encoder + linear projection for CTC.
    Input:  (B, T, F)   where F = pose_feature_dim
    Output: log-probs of shape (T, B, vocab_size) for CTC loss
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, bidirectional, vocab_size):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.vocab_size = vocab_size

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,          # input (B, T, F)
            bidirectional=bidirectional,
        )

        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)

        self.fc = nn.Linear(lstm_out_dim, vocab_size)

    def forward(self, x, x_lengths):
        """
        x:         (B, T, F) padded sequence
        x_lengths: (B,) lengths before padding
        """
        # Pack padded sequence for efficient LSTM computation
        packed = nn.utils.rnn.pack_padded_sequence(
            x, x_lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.lstm(packed)

        # Unpack back to (B, T, H)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        # Project to vocab
        logits = self.fc(out)          # (B, T, vocab_size)

        # CTC expects (T, B, C)
        log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
        return log_probs


#### 4.3. Instantiate and quick test cell

In [27]:
# Instantiate Model 1: BiLSTM + CTC

model1 = BiLSTM_CTC(
    input_dim=bilstm_config["input_dim"],
    hidden_dim=bilstm_config["hidden_dim"],
    num_layers=bilstm_config["num_layers"],
    dropout=bilstm_config["dropout"],
    bidirectional=bilstm_config["bidirectional"],
    vocab_size=bilstm_config["vocab_size"],
).to(device)

print(model1)

# Quick forward pass sanity check (reuse a small batch from train_loader)

model1.eval()
with torch.no_grad():
    batch = next(iter(train_loader))

    # Unpack the dictionary (if your loader returns a dict) OR the tuple
    # Assuming your loader returns a dictionary like Model 2 usually does:
    if isinstance(batch, dict):
        feats = batch["pose"].to(device)
        feat_lengths = batch["pose_lens"].to(device)
        # targets = batch["targets"] ... (not needed for forward pass check)
    else:
        # If it returns a tuple, unpack 5 items instead of 4
        feats, feat_lengths, targets, target_lengths, ids = batch  # Added 'ids'
        feats = feats.to(device)
        feat_lengths = feat_lengths.to(device)

    log_probs = model1(feats, feat_lengths)
    print("log_probs shape:", log_probs.shape)  # (T, B, vocab_size)


BiLSTM_CTC(
  (lstm): LSTM(344, 256, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (fc): Linear(in_features=512, out_features=678, bias=True)
)
log_probs shape: torch.Size([439, 16, 678])


#### 4.4 Optimizer and scheduler setup cell

In [28]:
# Optimizer & scheduler for Model 1 (Aligned with Model 2's batch-stepping)

lr = 1e-3
weight_decay = 1e-4

optimizer1 = torch.optim.AdamW(
    model1.parameters(),
    lr=lr,
    weight_decay=weight_decay,
)

# Use OneCycleLR to be compatible with 'train_one_epoch' stepping per batch
scheduler1 = torch.optim.lr_scheduler.OneCycleLR(
    optimizer1,
    max_lr=lr,
    epochs=40,  # Match your NUM_EPOCHS
    steps_per_epoch=len(train_loader),
    pct_start=0.3,
    anneal_strategy='cos'
)

print("Model 1 Optimizer & Scheduler configured (OneCycleLR)")

Model 1 Optimizer & Scheduler configured (OneCycleLR)


### 5. Model 1 Training + Evaluation

#### 5.1 Small helpers: prepare CTC targets and greedy decoder

In [29]:
# Helper: flatten padded target tensor into the 1D format that CTC expects
def pack_ctc_targets(targets_padded, target_lens):
    """
    targets_padded: [B, L_max] with padding at the end of each row.
    target_lens: [B] actual lengths (no pads).
    Returns:
        1D tensor of concatenated targets, length sum(target_lens).
    """
    pieces = []
    for i in range(targets_padded.size(0)):
        L = target_lens[i].item()
        pieces.append(targets_padded[i, :L])
    return torch.cat(pieces, dim=0)


def ctc_greedy_decode(logits, input_lengths=None, blank_index=CTC_BLANK_INDEX):
    """
    Decodes the logits (or log_probs) using greedy decoding.

    Args:
        logits: Tensor of shape (T, B, V+1)
        input_lengths: Optional tensor of shape (B,) - currently unused in this simple loop but good for API consistency
        blank_index: int index of the blank token
    """
    # Just treat logits as log_probs (argmax works the same on both)
    # shape: (T, B, C) -> (B, T)
    max_ids = logits.argmax(dim=-1).transpose(0, 1)

    results = []
    for seq in max_ids:
        prev = blank_index
        out = []
        for idx in seq.tolist():
            if idx != blank_index and idx != prev:
                out.append(idx)
            prev = idx
        results.append(out)
    return results


#### 5.2 WER Computation

In [30]:
def wer_one(ref_tokens, hyp_tokens):
    """
    Compute word error rate for a single pair (reference, hypothesis),
    given as lists of string tokens.
    """
    n = len(ref_tokens)
    m = len(hyp_tokens)

    # Classic edit distance DP
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                substitute = dp[i - 1][j - 1] + 1
                insert    = dp[i][j - 1] + 1
                delete    = dp[i - 1][j] + 1
                dp[i][j]  = min(substitute, insert, delete)

    if n == 0:
        return 0.0 if m == 0 else 1.0
    return dp[n][m] / n

In [31]:
def compute_wer_for_batch(batch, logits, blank_index):
    """
    Compute total word errors and total reference tokens over a batch.

    batch: dict from collate_fn with keys "pose_lens", "targets", "target_lens", "id"
    logits: tensor of shape [T, B, V+1] (before softmax is fine)
    blank_index: index of CTC blank token
    """
    # Decode predictions with greedy CTC
    pred_token_ids = ctc_greedy_decode(
        logits=logits,
        input_lengths=batch["pose_lens"],
        blank_index=blank_index,
    )  # list of length B, each a list of token ids

    total_err = 0.0
    total_ref = 0

    B = len(batch["id"])
    for i in range(B):
        # Reference tokens from target IDs (trim padding using target_lens)
        L_ref = batch["target_lens"][i].item()
        ref_ids = batch["targets"][i, :L_ref].cpu().tolist()
        ref_tokens = [id2token[t] for t in ref_ids if t < VOCAB_SIZE]

        # Hypothesis tokens from decoded IDs (ignore anything outside vocab)
        hyp_ids = [t for t in pred_token_ids[i] if t < VOCAB_SIZE]
        hyp_tokens = [id2token[t] for t in hyp_ids]

        total_err += wer_one(ref_tokens, hyp_tokens) * len(ref_tokens)
        total_ref += len(ref_tokens)

    return total_err, total_ref

In [32]:
def train_one_epoch(model, optimizer, data_loader, loss_fn, blank_index,
                    clip_grad=1.0, scheduler=None):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for batch in data_loader:
        pose = batch["pose"].to(device)           # [B, T, D]
        pose_lens = batch["pose_lens"].to(device) # [B]
        targets = batch["targets"].to(device)     # [B, Lmax]
        target_lens = batch["target_lens"].to(device)

        optimizer.zero_grad()

        logits = model(pose, pose_lens)           # [T, B, V+1]
        log_probs = logits.log_softmax(dim=-1)

        targets_flat = pack_ctc_targets(targets, target_lens)

        loss = loss_fn(
            log_probs,
            targets_flat,
            pose_lens,
            target_lens,
        )

        loss.backward()

        if clip_grad is not None:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

In [33]:
@torch.no_grad()
def evaluate_ctc_model(model, data_loader, loss_fn, blank_index, max_batches=None):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    total_err = 0.0
    total_ref = 0

    for b_idx, batch in enumerate(data_loader):
        pose = batch["pose"].to(device)
        pose_lens = batch["pose_lens"].to(device)
        targets = batch["targets"].to(device)
        target_lens = batch["target_lens"].to(device)

        logits = model(pose, pose_lens)
        log_probs = logits.log_softmax(dim=-1)
        targets_flat = pack_ctc_targets(targets, target_lens)

        loss = loss_fn(
            log_probs,
            targets_flat,
            pose_lens,
            target_lens,
        )

        # WER: get total errors and refs for this batch
        batch_err, batch_ref = compute_wer_for_batch(batch, logits.cpu(), blank_index)

        total_loss += loss.item()
        total_batches += 1
        total_err += batch_err
        total_ref += batch_ref

        if (max_batches is not None) and (b_idx + 1 >= max_batches):
            break

    avg_loss = total_loss / max(total_batches, 1)
    avg_wer = total_err / max(total_ref, 1)
    return avg_loss, avg_wer

#### 5.3 Full training Loop

In [ ]:
# Training loop for Model 1: BiLSTM + CTC

NUM_EPOCHS = 40
EVAL_EVERY = 1
checkpoint_path = "/content/drive/MyDrive/best_model1.pt"  # Path to save the best model

best_dev_wer = float("inf")
best_state_dict = None

print(f"Starting training... Best model will be saved to '{checkpoint_path}'")

for epoch in range(1, NUM_EPOCHS + 1):
    # TRAINING
    train_loss = train_one_epoch(
        model=model1,
        optimizer=optimizer1,
        data_loader=train_loader,
        loss_fn=ctc_loss_fn,
        blank_index=CTC_BLANK_INDEX,
        clip_grad=1.0,
        scheduler=scheduler1,
    )

    # EVALUATION
    dev_loss, dev_wer = evaluate_ctc_model(
        model=model1,
        data_loader=dev_loader,
        loss_fn=ctc_loss_fn,
        blank_index=CTC_BLANK_INDEX,
        max_batches=None,
    )

    print(f"Epoch {epoch:02d} | train CTC loss = {train_loss:.4f} | "
          f"dev CTC loss = {dev_loss:.4f} | dev WER = {dev_wer:.4f}")

    # SAVE IF BEST
    if dev_wer < best_dev_wer:
        best_dev_wer = dev_wer
        best_state_dict = model1.state_dict()
        # Save to file immediately
        torch.save(best_state_dict, checkpoint_path)
        print(f" --> New best dev WER! Model saved to {checkpoint_path}")

    # OPTIONAL: Save every epoch (e.g., for resuming later)
    # torch.save(model1.state_dict(), f"model1_epoch_{epoch}.pth")

# After training, ensure we have the best version loaded
if best_state_dict is not None:
    model1.load_state_dict(best_state_dict)
    print(f"\nLoaded best BiLSTM model from memory with dev WER: {best_dev_wer:.4f}")


Starting training... Best model will be saved to '/content/drive/MyDrive/best_model1.pt'
Epoch 01 | train CTC loss = 32.6290 | dev CTC loss = 5.7668 | dev WER = 0.9256
 --> New best dev WER! Model saved to /content/drive/MyDrive/best_model1.pt
Epoch 02 | train CTC loss = 5.7093 | dev CTC loss = 5.6487 | dev WER = 0.9256
Epoch 03 | train CTC loss = 5.6122 | dev CTC loss = 5.5622 | dev WER = 0.9240
 --> New best dev WER! Model saved to /content/drive/MyDrive/best_model1.pt
Epoch 04 | train CTC loss = 5.5308 | dev CTC loss = 5.5210 | dev WER = 0.9080
 --> New best dev WER! Model saved to /content/drive/MyDrive/best_model1.pt
Epoch 05 | train CTC loss = 5.4318 | dev CTC loss = 5.3545 | dev WER = 0.9097
Epoch 06 | train CTC loss = 5.2691 | dev CTC loss = 5.2736 | dev WER = 0.9146
Epoch 07 | train CTC loss = 5.1326 | dev CTC loss = 5.0546 | dev WER = 0.9185
Epoch 08 | train CTC loss = 4.9781 | dev CTC loss = 4.9659 | dev WER = 0.9111
Epoch 09 | train CTC loss = 4.8038 | dev CTC loss = 4.6597

KeyboardInterrupt: 

###### **Note on Early Stopping & Model Performance**

---

**Manual Interruption at Epoch 37**

Training of the BiLSTM model was manually stopped at **Epoch 37** after observing a performance plateau across several runs. Given the limited GPU budget available in Colab, continuing training offered minimal improvement. Redirecting compute resources toward more advanced architectures—such as the Conformer—provides significantly better returns.

---

**Why the BiLSTM Baseline Underperformed**

Despite reaching a reasonable loss, the BiLSTM consistently failed to achieve competitive Word Error Rates (WER). The key architectural limitations include:

**1. Loss of Spatial Structure**
The pose input is inherently **3D skeletal data**, where relationships between joints (e.g., hand-to-face distance) are critical.  
Flattening the pose into a vector at each time-step destroys this geometric structure.  
As a result, the BiLSTM treats the data as an unordered set of numbers instead of a coordinated skeleton.

**2. Limited Long-Range Dependency**
High-frame-rate pose sequences span **hundreds of time-steps**.  
Although LSTMs are designed for sequential tasks, their ability to carry gradient signals over very long sequences is limited.  
This makes it difficult for the model to capture the full temporal context of signs that evolve over many frames.

**3. Lack of Local Feature Extraction**
Standard LSTMs do not have a mechanism for capturing **local motion patterns**—such as brief directional hand movements—before processing the entire sequence.  
Later architectures (e.g., Conformer) integrate convolutional modules that handle these local features much more effectively.

---

**Next Steps**

We now move forward to evaluating the **Conformer (Model 2)**  
(and Models 3 and 4 in a separate notebook).  

The Conformer architecture directly addresses the BiLSTM’s weaknesses by combining:

- **Convolutional layers** → Extract local motion cues  
- **Transformer-based attention** → Capture long-range temporal relationships  

This makes it substantially more suitable for pose-based Saudi Sign Language recognition.

---

#### 5.4 Testing Results of each model | Samples from the Validation \ Test

##### 5.4.1 Collect dev samples (for qualitative examples)

In [46]:
# Helper functions for generating predictions and samples

import pandas as pd
import random

def collect_sample_predictions(model, data_loader, df, n_samples=10, max_batches=10):
    """
    Collect a few (id, reference, prediction) triplets from a loader for inspection.
    df: DataFrame (train/dev) that has columns ['id', 'gloss'].
    """
    model.eval()
    samples = []
    seen_ids = set()

    with torch.no_grad():
        for b_idx, batch in enumerate(data_loader):
            pose = batch["pose"].to(device)
            pose_lens = batch["pose_lens"].to(device)
            ids = batch["id"]                     # list/array of sample ids

            # Forward + decode
            logits = model(pose, pose_lens)       # [T, B, V+1]
            pred_token_ids = ctc_greedy_decode(
                logits=logits.cpu(),
                blank_index=CTC_BLANK_INDEX,
            )

            B = len(ids)
            for i in range(B):
                sid = ids[i]
                if sid in seen_ids:
                    continue
                seen_ids.add(sid)

                # Reference gloss from dev_df
                ref_row = df[df["id"] == sid]
                if len(ref_row) > 0:
                    ref_sent = ref_row["gloss"].values[0]
                else:
                    # Fallback: reconstruct from target IDs in batch
                    # This branch hits if df doesn't have the id, or for test set
                    if "targets" in batch:
                        L_ref = batch["target_lens"][i].item()
                        ref_ids = batch["targets"][i, :L_ref].cpu().tolist()
                        ref_tokens = [id2token[t] for t in ref_ids if t < VOCAB_SIZE]
                        ref_sent = " ".join(ref_tokens)
                    else:
                        ref_sent = ""

                # Hypothesis gloss from decoded IDs
                hyp_ids = [t for t in pred_token_ids[i] if t < VOCAB_SIZE]
                hyp_tokens = [id2token[t] for t in hyp_ids]
                hyp_sent = " ".join(hyp_tokens)

                samples.append({"id": sid, "reference": ref_sent, "prediction": hyp_sent})

                if len(samples) >= n_samples:
                    break

            if len(samples) >= n_samples or (max_batches is not None and b_idx + 1 >= max_batches):
                break

    return samples

def generate_submission_csv(model, data_loader, out_path,
                            vocab_size=VOCAB_SIZE,
                            blank_index=CTC_BLANK_INDEX):
    """
    Run CTC model on a loader that yields:
      batch["pose"]      : [B, T, D]
      batch["pose_lens"] : [B]
      batch["id"]        : sample ids (matching dev/test csv)
    and save predictions as a CSV with columns ['id', 'gloss'].
    """
    model.eval()
    all_ids = []
    all_gloss = []

    with torch.no_grad():
        for batch in data_loader:
            pose = batch["pose"].to(device)
            pose_lens = batch["pose_lens"].to(device)
            ids = batch["id"]

            logits = model(pose, pose_lens)  # [T, B, V+1]
            pred_token_ids = ctc_greedy_decode(
                logits=logits.cpu(),
                blank_index=blank_index,
            )

            for i, sid in enumerate(ids):
                hyp_ids = [t for t in pred_token_ids[i] if t < vocab_size]
                hyp_tokens = [id2token[t] for t in hyp_ids]
                gloss = " ".join(hyp_tokens)
                all_ids.append(sid)
                all_gloss.append(gloss)

    sub_df = pd.DataFrame({"id": all_ids, "gloss": all_gloss})
    sub_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved predictions to {out_path} with shape {sub_df.shape}")
    return sub_df


##### 5.4.1 Generate submission CSV (dev or test)

In [47]:
def generate_submission_csv(model, data_loader, out_path,
                            vocab_size=VOCAB_SIZE,
                            blank_index=CTC_BLANK_INDEX):
    model.eval()
    all_ids, all_gloss = [], []
    with torch.no_grad():
        for batch in data_loader:
            pose = batch["pose"].to(device)
            pose_lens = batch["pose_lens"].to(device)
            ids = batch["id"]
            logits = model(pose, pose_lens)
            pred_token_ids = ctc_greedy_decode(
                logits=logits.cpu(),
                blank_index=blank_index,
            )
            for i, sid in enumerate(ids):
                hyp_ids = [t for t in pred_token_ids[i] if t < vocab_size]
                hyp_tokens = [id2token[t] for t in hyp_ids]
                gloss = " ".join(hyp_tokens)
                all_ids.append(sid)
                all_gloss.append(gloss)
    sub_df = pd.DataFrame({"id": all_ids, "gloss": all_gloss})
    sub_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved predictions to {out_path} with shape {sub_df.shape}")
    return sub_df

##### 5.4.2 reload, evaluate, show samples, write CSVs

In [ ]:
bilstm_ckpt_path = "/content/drive/MyDrive/best_model1.pt"
model1.load_state_dict(torch.load(bilstm_ckpt_path, map_location=device))
model1.to(device)

bilstm_dev_loss, bilstm_dev_wer = evaluate_ctc_model(
    model=model1,
    data_loader=dev_loader,
    loss_fn=ctc_loss_fn,
    blank_index=CTC_BLANK_INDEX,
)
print(f"[BiLSTM] Dev CTC loss = {bilstm_dev_loss:.4f} | Dev WER = {bilstm_dev_wer:.4f}")

bilstm_samples = collect_sample_predictions(model1, dev_loader, dev_df, n_samples=10, max_batches=20)
# print samples (for your report)

bilstm_dev_csv_path = "/content/drive/MyDrive/bilstm_dev_predictions.csv"
generate_submission_csv(model1, dev_loader, bilstm_dev_csv_path)

bilstm_test_csv_path = "/content/drive/MyDrive/bilstm_test_predictions.csv"
generate_submission_csv(model1, test_loader, bilstm_test_csv_path)

### 5.5 Error Analysis — the BiLSTM baseline

The training log above is the analysis. Two things in it explain why this architecture was abandoned rather than tuned further.

**It fits the training set without the representation generalising.** By epoch 34 the train CTC loss has fallen from 32.63 to 1.02 while the dev loss sits at 1.84 and dev WER at 66.62%. The model is learning the training data; what it learns does not transfer.

**Progress had gone flat.** WER improves from 92.56% to 66.62% over 34 epochs, but the last ten contribute barely three points and the curve is asymptotic rather than descending. Continuing would have bought a little more at a cost the Colab budget could not justify against trying a stronger encoder.

Both symptoms trace to the two structural limits described above: flattening each frame into a vector discards the geometric relationships between joints, and a single recurrent state is a narrow channel for relating a handshape early in a sentence to a movement much later.

The detailed per-error breakdown, deletions against substitutions against insertions, is carried out on the two models that were actually trained to completion: the Conformer with CTC below, and the Seq2Seq variant in `03-conformer-seq2seq.ipynb`.

> **Note on the checkpoint reload cell below.** It was not re-executed in this saved session, so it carries no stored output. The BiLSTM's reported result is the best dev WER recorded during training, 66.62% at epoch 34.

### 6. Model 2: Conformer + CTC

The Conformer architecture improves upon the RNN baseline by combining:
1.  **Convolutional Module:** To extract local temporal features (e.g., rapid hand movements).
2.  **Self-Attention Module:** To capture long-range global dependencies across the entire sentence.
This hybrid approach allows the model to "see" both fine-grained details and the overall sentence structure simultaneously.


#### 6.1. Model configuration

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Dynamically compute input dimension from a sample
sample_feat = pose_to_feature_sequence(get_pose_array(train_df["id"].iloc[0]))
INPUT_DIM = sample_feat.shape[-1]
print(f"Input dimension: {INPUT_DIM}")

# Vocabulary setup
print(f"Vocab size: {VOCAB_SIZE}, CTC blank index: {CTC_BLANK_INDEX}")

# Conformer hyperparameters (tuned for Isharah based on published results)
D_MODEL = 256           # Model dimension
N_HEADS = 4             # Number of attention heads
N_LAYERS = 6            # Number of Conformer blocks
CONV_KERNEL_SIZE = 31   # Depthwise conv kernel (must be odd)
FFN_EXPANSION = 4       # Feed-forward expansion factor
DROPOUT = 0.1           # Dropout rate


Using device: cuda
Input dimension: 344
Vocab size: 677, CTC blank index: 677


#### 6.2. Conformer Block Implementation

In [25]:
class ConformerBlock(nn.Module):
    """
    Single Conformer block following architecture:
    FFN(1/2) -> Multi-Head Self-Attention -> Convolution -> FFN(1/2) -> LayerNorm
    """
    def __init__(self, d_model, n_heads, conv_kernel_size, ffn_expansion=4, dropout=0.1):
        super().__init__()

        # First half of feed-forward module
        self.ffn1 = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * ffn_expansion),
            nn.SiLU(),  # Swish activation
            nn.Dropout(dropout),
            nn.Linear(d_model * ffn_expansion, d_model),
            nn.Dropout(dropout),
        )

        # Multi-head self-attention module
        self.attn_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attn_dropout = nn.Dropout(dropout)

        # Convolution module (depthwise separable)
        self.conv_norm = nn.LayerNorm(d_model)
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, d_model * 2, kernel_size=1),  # Pointwise expansion
            nn.GLU(dim=1),  # Gated Linear Unit
            nn.Conv1d(d_model, d_model, kernel_size=conv_kernel_size,
                     padding=(conv_kernel_size - 1) // 2, groups=d_model),  # Depthwise
            nn.BatchNorm1d(d_model),
            nn.SiLU(),
            nn.Conv1d(d_model, d_model, kernel_size=1),  # Pointwise projection
            nn.Dropout(dropout),
        )

        # Second half of feed-forward module
        self.ffn2 = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * ffn_expansion),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ffn_expansion, d_model),
            nn.Dropout(dropout),
        )

        # Final layer norm
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        """
        Args:
            x: [B, T, D]
            mask: [B, T] boolean mask (True = valid, False = padding)
        Returns:
            [B, T, D]
        """
        # 1. First FFN with residual (half-step)
        x = x + 0.5 * self.ffn1(x)

        # 2. Multi-head self-attention with residual
        x_norm = self.attn_norm(x)


        attn_out, _ = self.self_attn(x_norm, x_norm, x_norm,
                                     key_padding_mask=~mask if mask is not None else None)

        x = x + self.attn_dropout(attn_out)

        # 3. Convolution module with residual
        x_norm = self.conv_norm(x)
        x_conv = x_norm.transpose(1, 2)  # [B, T, D] -> [B, D, T]
        x_conv = self.conv(x_conv)
        x_conv = x_conv.transpose(1, 2)  # [B, D, T] -> [B, T, D]
        x = x + x_conv

        # 4. Second FFN with residual (half-step)
        x = x + 0.5 * self.ffn2(x)

        # 5. Final layer norm
        x = self.norm(x)

        return x


#### 6.3. Full Conformer + CTC Model


In [26]:
class ConformerCTCModel(nn.Module):
    """
    Complete Conformer encoder with CTC output layer for CSLR.
    """
    def __init__(
        self,
        input_dim,
        d_model,
        n_heads,
        n_layers,
        vocab_size,
        ctc_blank_index,
        conv_kernel_size=31,
        ffn_expansion=4,
        dropout=0.1,
    ):
        super().__init__()

        self.input_dim = input_dim
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.ctc_blank_index = ctc_blank_index

        # Input projection (linear layer to map input_dim -> d_model)
        self.input_proj = nn.Linear(input_dim, d_model)

        # Sinusoidal positional encoding
        self.pos_encoding = PositionalEncoding(d_model, dropout, max_len=1000)

        # Stack of Conformer blocks
        self.conformer_blocks = nn.ModuleList([
            ConformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                conv_kernel_size=conv_kernel_size,
                ffn_expansion=ffn_expansion,
                dropout=dropout,
            )
            for _ in range(n_layers)
        ])

        # Output projection to vocab + blank
        self.fc_out = nn.Linear(d_model, vocab_size + 1)

    def forward(self, x, lengths):
        """
        Args:
            x: [B, T, input_dim] - padded pose features
            lengths: [B] - actual sequence lengths
        Returns:
            logits: [T, B, vocab_size+1] - CTC logits (time-major)
        """
        B, T, _ = x.shape

        # Create padding mask
        mask = torch.arange(T, device=x.device).unsqueeze(0) < lengths.unsqueeze(1)  # [B, T]

        # Input projection
        x = self.input_proj(x)  # [B, T, d_model]

        # Add positional encoding
        x = self.pos_encoding(x)

        # Pass through Conformer blocks
        for block in self.conformer_blocks:
            x = block(x, mask=mask)

        # Output projection
        logits = self.fc_out(x)  # [B, T, vocab_size+1]

        # CTC expects [T, B, vocab_size+1]
        logits = logits.transpose(0, 1)

        return logits


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (from Attention Is All You Need)."""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: [B, T, d_model]
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


#### 6.4 Instantiate and Test the Model

In [27]:
# Create the Conformer model
conformer_model = ConformerCTCModel(
    input_dim=INPUT_DIM,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    vocab_size=VOCAB_SIZE,
    ctc_blank_index=CTC_BLANK_INDEX,
    conv_kernel_size=CONV_KERNEL_SIZE,
    ffn_expansion=FFN_EXPANSION,
    dropout=DROPOUT,
).to(device)

# Print model summary
total_params = sum(p.numel() for p in conformer_model.parameters())
trainable_params = sum(p.numel() for p in conformer_model.parameters() if p.requires_grad)
print(f"\nConformer Model Summary:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Quick forward pass test
batch = next(iter(train_loader))
pose = batch["pose"].to(device)
pose_lens = batch["pose_lens"].to(device)

with torch.no_grad():
    logits = conformer_model(pose, pose_lens)
    print(f"\nForward pass test:")
    print(f"Input shape: {pose.shape} [B, T, D]")
    print(f"Output shape: {logits.shape} [T, B, vocab_size+1]")
    print(f"Expected vocab+blank size: {VOCAB_SIZE + 1}")


Conformer Model Summary:
Total parameters: 9,400,230
Trainable parameters: 9,400,230

Forward pass test:
Input shape: torch.Size([16, 341, 344]) [B, T, D]
Output shape: torch.Size([341, 16, 678]) [T, B, vocab_size+1]
Expected vocab+blank size: 678


#### 6.5 Optimizer and Scheduler Setup

In [28]:
# CTC loss function
ctc_loss_fn = nn.CTCLoss(
    blank=CTC_BLANK_INDEX,
    reduction="mean",
    zero_infinity=True,
)

# Optimizer: AdamW with weight decay for better generalization
LEARNING_RATE = 3e-4  # Lower LR for Transformers
WEIGHT_DECAY = 1e-4

conformer_optimizer = torch.optim.AdamW(
    conformer_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98),  # Standard for Transformers
    eps=1e-9,
)

# Learning rate scheduler with warmup
NUM_WARMUP_EPOCHS = 5

conformer_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    conformer_optimizer,
    max_lr=LEARNING_RATE,
    epochs=40,  # Match your NUM_EPOCHS
    steps_per_epoch=len(train_loader),
    pct_start = 0.2,  # Warmup fraction - PREVIOUSLY = NUM_WARMUP_EPOCHS / 40
    anneal_strategy='cos',
)

print("Optimizer and scheduler configured.")

Optimizer and scheduler configured.


### 7. Model 2 Training + Evaluation

#### 7.1. Small helpers: prepare CTC targets and greedy decoder

In [40]:
# Helper: flatten padded target tensor into the 1D format that CTC expects
def pack_ctc_targets(targets_padded, target_lens):
    """
    targets_padded: [B, L_max] with padding at the end of each row.
    target_lens: [B] actual lengths (no pads).
    Returns:
        1D tensor of concatenated targets, length sum(target_lens).
    """
    pieces = []
    for i in range(targets_padded.size(0)):
        L = target_lens[i].item()
        pieces.append(targets_padded[i, :L])
    return torch.cat(pieces, dim=0)

def ctc_greedy_decode(logits, input_lengths=None, blank_index=CTC_BLANK_INDEX):
    """
    logits: (T, B, C)
    input_lengths: optional (B,) - real lengths of sequences in batch
    """
    # max_ids shape: (B, T)
    max_ids = logits.argmax(dim=-1).transpose(0, 1)
    B, T_max = max_ids.shape

    results = []
    for b in range(B):
        # Determine effective length for this sample
        if input_lengths is not None:
            seq_len = input_lengths[b].item()
        else:
            seq_len = T_max

        # Get the sequence up to seq_len
        seq = max_ids[b, :seq_len].tolist()

        prev = blank_index
        out = []
        for idx in seq:
            if idx != blank_index and idx != prev:
                out.append(idx)
            prev = idx
        results.append(out)
    return results


#### 7.2. WER computation

In [30]:
def wer_one(ref_tokens, hyp_tokens):
    """
    Compute word error rate for a single pair (reference, hypothesis),
    given as lists of string tokens.
    """
    n = len(ref_tokens)
    m = len(hyp_tokens)

    # Classic edit distance DP
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                substitute = dp[i - 1][j - 1] + 1
                insert    = dp[i][j - 1] + 1
                delete    = dp[i - 1][j] + 1
                dp[i][j]  = min(substitute, insert, delete)

    if n == 0:
        return 0.0 if m == 0 else 1.0
    return dp[n][m] / n

In [31]:
def compute_wer_for_batch(batch, logits, blank_index):
    """
    Compute WER over a batch.
    batch: dict from collate_batch.
    logits: [T, B, V+1]
    """
    # Decode predictions
    pred_token_ids = ctc_greedy_decode(
        logits=logits,
        input_lengths=batch["pose_lens"],
        blank_index=blank_index,
    )

    total_err = 0.0
    total_ref = 0

    B = len(batch["id"])
    for i in range(B):
        # Reference gloss tokens (strings)
        ref_sentence = str(batch["targets"][i, : batch["target_lens"][i]].cpu().numpy().tolist())
        # We'll re-tokenize from original gloss text to keep it simple
        # (safer than decoding IDs with pads etc.)
        ref_text = str(batch["targets"][i, : batch["target_lens"][i]])
        # Instead, better to use the original gloss string from the DataFrame if we keep it;
        # but here we'll reconstruct from IDs:
        ref_ids = batch["targets"][i, : batch["target_lens"][i]].cpu().tolist()
        ref_tokens = [id2token[idx] for idx in ref_ids]

        # Hypothesis tokens from decoded IDs, skipping anything outside vocab range
        hyp_ids = [tid for tid in pred_token_ids[i] if tid < VOCAB_SIZE]
        hyp_tokens = [id2token[idx] for idx in hyp_ids]

        total_err += wer_one(ref_tokens, hyp_tokens) * len(ref_tokens)
        total_ref += len(ref_tokens)

    return total_err, total_ref

#### 7.3. Training and evaluation loops for model 2


In [32]:
def train_one_epoch(model, optimizer, data_loader, loss_fn, blank_index,
                    clip_grad=1.0, scheduler=None):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for batch in data_loader:
        pose = batch["pose"].to(device)           # [B, T, D]
        pose_lens = batch["pose_lens"].to(device) # [B]
        targets = batch["targets"].to(device)     # [B, Lmax]
        target_lens = batch["target_lens"].to(device)

        optimizer.zero_grad()

        logits = model(pose, pose_lens)           # [T, B, V+1]
        log_probs = logits.log_softmax(dim=-1)

        targets_flat = pack_ctc_targets(targets, target_lens)

        loss = loss_fn(
            log_probs,
            targets_flat,
            pose_lens,
            target_lens,
        )

        loss.backward()

        if clip_grad is not None:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

In [33]:
@torch.no_grad()
def evaluate_ctc_model(model, data_loader, loss_fn, blank_index, max_batches=None):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    total_err = 0.0
    total_ref = 0

    for b_idx, batch in enumerate(data_loader):
        pose = batch["pose"].to(device)
        pose_lens = batch["pose_lens"].to(device)
        targets = batch["targets"].to(device)
        target_lens = batch["target_lens"].to(device)

        logits = model(pose, pose_lens)
        log_probs = logits.log_softmax(dim=-1)
        targets_flat = pack_ctc_targets(targets, target_lens)

        loss = loss_fn(
            log_probs,
            targets_flat,
            pose_lens,
            target_lens,
        )

        # WER: get total errors and refs for this batch
        batch_err, batch_ref = compute_wer_for_batch(batch, logits.cpu(), blank_index)

        total_loss += loss.item()
        total_batches += 1
        total_err += batch_err
        total_ref += batch_ref

        if (max_batches is not None) and (b_idx + 1 >= max_batches):
            break

    avg_loss = total_loss / max(total_batches, 1)
    avg_wer = total_err / max(total_ref, 1)
    return avg_loss, avg_wer

#### Full Quick sanity cell (run before training)

In [34]:
batch = next(iter(train_loader))
pose = batch["pose"].to(device)
pose_lens = batch["pose_lens"].to(device)

with torch.no_grad():
    logits = conformer_model(pose, pose_lens)

print("Pose:", pose.shape)        # [B, T, D]
print("Logits:", logits.shape)    # [T, B, VOCAB_SIZE+1]
print("VOCAB_SIZE+1:", VOCAB_SIZE + 1)

Pose: torch.Size([16, 493, 344])
Logits: torch.Size([493, 16, 678])
VOCAB_SIZE+1: 678


#### 7.4. Full training loop

In [32]:
NUM_EPOCHS = 40
EVAL_EVERY = 1
checkpoint_path = "/content/drive/MyDrive/best_model2.pt"


best_dev_wer = float("inf")
best_state_dict = None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(
        model=conformer_model,
        optimizer=conformer_optimizer,
        data_loader=train_loader,
        loss_fn=ctc_loss_fn,
        blank_index=CTC_BLANK_INDEX,
        clip_grad=1.0,
        scheduler=conformer_scheduler,  # Step per batch
    )

    dev_loss, dev_wer = evaluate_ctc_model(
        model=conformer_model,
        data_loader=dev_loader,
        loss_fn=ctc_loss_fn,
        blank_index=CTC_BLANK_INDEX,
        max_batches=None,
    )

    print(f"Epoch {epoch:02d} | train CTC loss = {train_loss:.4f} | "
          f"dev CTC loss = {dev_loss:.4f} | dev WER = {dev_wer:.4f} | "
          f"LR = {conformer_scheduler.get_last_lr()[0]:.2e}")

    if dev_wer < best_dev_wer:
        best_dev_wer = dev_wer
        best_state_dict = conformer_model.state_dict()
        torch.save(best_state_dict, checkpoint_path)
        print("  -> New best dev WER, saving model checkpoint.")

if best_state_dict is not None:
    conformer_model.load_state_dict(best_state_dict)
    print(f"\nLoaded best Conformer model with dev WER: {best_dev_wer:.4f}")


Epoch 01 | train CTC loss = 20.8047 | dev CTC loss = 5.8877 | dev WER = 1.0000 | LR = 2.30e-05
  -> New best dev WER, saving model checkpoint.
Epoch 02 | train CTC loss = 5.7673 | dev CTC loss = 5.6646 | dev WER = 0.9194 | LR = 5.42e-05
  -> New best dev WER, saving model checkpoint.
Epoch 03 | train CTC loss = 5.5772 | dev CTC loss = 5.4696 | dev WER = 0.9095 | LR = 1.01e-04
  -> New best dev WER, saving model checkpoint.
Epoch 04 | train CTC loss = 5.3578 | dev CTC loss = 5.2766 | dev WER = 0.9005 | LR = 1.56e-04
  -> New best dev WER, saving model checkpoint.
Epoch 05 | train CTC loss = 5.0780 | dev CTC loss = 4.8771 | dev WER = 0.8729 | LR = 2.11e-04
  -> New best dev WER, saving model checkpoint.
Epoch 06 | train CTC loss = 4.7013 | dev CTC loss = 4.4191 | dev WER = 0.8531 | LR = 2.58e-04
  -> New best dev WER, saving model checkpoint.
Epoch 07 | train CTC loss = 4.2090 | dev CTC loss = 3.8174 | dev WER = 0.7852 | LR = 2.89e-04
  -> New best dev WER, saving model checkpoint.
Epoch

### 7.5 Testing Results of each model | Samples from the Validation \ Test

In [54]:
# 1) Reload best Conformer weights from Drive
conformer_ckpt_path = "/content/drive/MyDrive/best_model2.pt"
print(f"Loading Conformer weights from: {conformer_ckpt_path}")
conformer_model.load_state_dict(torch.load(conformer_ckpt_path, map_location=device))
conformer_model.to(device)

# 2) Quantitative dev evaluation
conf_dev_loss, conf_dev_wer = evaluate_ctc_model(
    model=conformer_model,
    data_loader=dev_loader,
    loss_fn=ctc_loss_fn,
    blank_index=CTC_BLANK_INDEX,
    max_batches=None,
)
print(f"[Conformer] Dev CTC loss = {conf_dev_loss:.4f} | Dev WER = {conf_dev_wer:.4f}")

# 3) A few qualitative examples from dev
conf_samples = collect_sample_predictions(
    model=conformer_model,
    data_loader=dev_loader,
    df=dev_df,
    n_samples=10,
    max_batches=20,
)

print("\n[Conformer] Sample predictions on dev set:\n")
for s in conf_samples:
    print(f"ID : {s['id']}")
    print(f"REF: {s['reference']}")
    print(f"PRED:{s['prediction']}")
    print("-" * 80)

# 4) Generate full dev predictions CSV (for Phase-1-style file)
conf_dev_csv_path = "/content/drive/MyDrive/conformer_dev_predictions.csv"
print("\nGenerating full dev predictions CSV for Conformer...")
_ = generate_submission_csv(
    model=conformer_model,
    data_loader=dev_loader,
    out_path=conf_dev_csv_path,
)

# 5) Generate full test predictions CSV (for Phase-2-style file)
# Requires test_loader from the steps we added earlier
conf_test_csv_path = "/content/drive/MyDrive/conformer_test_predictions.csv"
print("\nGenerating full test predictions CSV for Conformer...")
_ = generate_submission_csv(
    model=conformer_model,
    data_loader=test_loader,
    out_path=conf_test_csv_path,
)

print("\n[Conformer] Done generating dev/test predictions.")


Loading Conformer weights from: /content/drive/MyDrive/best_model2.pt
[Conformer] Dev CTC loss = 0.4856 | Dev WER = 0.1304

[Conformer] Sample predictions on dev set:

ID : 02_0001
REF: سوال هو
PRED:سوال
--------------------------------------------------------------------------------
ID : 02_0002
REF: هو معلم لغه اشاره
PRED:هو معرفه لغه اشاره
--------------------------------------------------------------------------------
ID : 02_0003
REF: استفهام هو معلم هو
PRED:هو معلم هو
--------------------------------------------------------------------------------
ID : 02_0004
REF: هو معلم لا انا مدرسه
PRED:هو معلم لا انا مدرسه
--------------------------------------------------------------------------------
ID : 02_0005
REF: هو سوال
PRED:هو سوال
--------------------------------------------------------------------------------
ID : 02_0006
REF: هو صديق مدرسه
PRED:هو صديق مدرسه
--------------------------------------------------------------------------------
ID : 02_0007
REF: استفهام هو صديق مدرسه
PR

### 7.6 Error Analysis

#### **Conformer Model Evaluation Summary**

The Conformer model achieved a **Development WER of 13.04%**, indicating strong and stable performance. Using the prediction outputs (949 development samples) and analyzing multiple error cases, the following insights were observed:

---

##### **1. Statistical Overview**

**Prediction Lengths**  
- The model outputs sentences with a **mean length of 4.5 tokens** (maximum 12).  
- This closely matches the natural phrase lengths in the dataset.  
- No signs of CTC collapse (too few tokens) or repetition loops (overgeneration).

**Vocabulary Bias**  
The most frequently predicted words are common in sign-language discourse:  
- **"انا"** — 383 occurrences  
- **"هو"** — 207 occurrences  
- **"سوال"** — 113 occurrences  

This distribution reflects realistic linguistic frequency rather than model instability.

---

##### **2. Error Categories**

###### **A. Deletions (Most Common)**
The model often removes short markers or light function words, typically at sentence boundaries.

- **Example (02_0001)**  
  - REF: `سوال هو`  
  - PRED: `سوال`  
  - Missing token: **هو**

- **Example (02_0003)**  
  - REF: `استفهام هو معلم هو`  
  - PRED: `هو معلم هو`  
  - Missing token: **استفهام** (question marker)

**Cause:**  
Short signs are harder to distinguish due to co-articulation and blurred motion in continuous signing, making CTC alignment less reliable for these tokens.

---

###### **B. Substitutions (Semantic / Visual Confusions)**

- **Example (02_0002)**  
  - REF: `هو معلم لغه اشاره`  
  - PRED: `هو معرفه لغه اشاره`  
  - Substitution: **معلم → معرفه**

**Cause:**  
Both signs may share similar handshapes or movement patterns in the pose data, leading to visually plausible confusion.

---

###### **C. Length Robustness**

The model maintains high accuracy even for longer sentences.

- **Example (02_0004)**  
  - REF: `هو معلم لا انا مدرسه`  
  - PRED: (correct 5-token prediction)

- **Example (02_0010)**  
  - REF: `عمر اربع سن عمر`  
  - PRED: (correct 4-token prediction)

This indicates strong temporal modeling and stable decoding behavior.

---

##### **Conclusion**

The Conformer model clearly outperforms the BiLSTM baseline.  
Its errors are mostly minor—primarily deletions of low-impact words—rather than severe semantic hallucinations. The close match in prediction lengths between **Dev (mean 4.53)** and **Test (mean 4.37)** further confirms that the model generalizes well and is reliable.

---
